# Kubernetes RAG System Evaluation with RAGAS

This notebook evaluates the Kubernetes RAG system using the RAGAS framework, comparing base retrieval methods with advanced retrieval techniques.

## Evaluation Metrics

We'll evaluate the following metrics:
- **Faithfulness**: Measures how factually accurate the generated answer is based on the given context
- **Response Relevancy**: Measures how relevant the generated answer is to the given prompt
- **Context Precision**: Measures the proportion of relevant items in the retrieved context
- **Context Recall**: Measures the proportion of relevant context that was successfully retrieved


## Setup and Imports


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add the k8s_rag module to path
sys.path.append('.')

from k8s_rag.vector_db.vector_store import K8sDocVectorStore
from k8s_rag.vector_db.data_loader import K8sDocumentationLoader
from k8s_rag.agents.base_agent import K8sBaseRAGAgent
from k8s_rag.retrieval.advanced_retrievers import K8sAdvancedRetrieverFactory, RetrieverType
from k8s_rag.evaluation.evaluator import K8sRAGEvaluator
from k8s_rag.utils.config import setup_environment, get_config

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")


## Environment Setup


In [ ]:
# Set up API keys (you'll need to provide these)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("Enter your OpenAI API key: ")

# Optional: Cohere API key for better reranking
if not os.getenv("COHERE_API_KEY"):
    cohere_key = input("Enter your Cohere API key (optional, press enter to skip): ")
    if cohere_key.strip():
        os.environ["COHERE_API_KEY"] = cohere_key

# Verify environment setup
if setup_environment():
    print("✅ Environment setup successful!")
    config = get_config()
    print(f"📊 Configuration: {config}")
else:
    print("❌ Environment setup failed!")


## Data Loading and System Initialization


In [ ]:
# Initialize vector store
print("🔧 Initializing vector store...")
vector_store = K8sDocVectorStore()

# Load Kubernetes documentation
data_dir = Path("./data")
if not data_dir.exists():
    print(f"❌ Data directory not found: {data_dir}")
    print("Please ensure the Kubernetes documentation is available in the 'data' directory")
else:
    print(f"📚 Loading documentation from {data_dir}...")
    loader = K8sDocumentationLoader(data_dir)
    loader.load_all_data(vector_store)
    
    # Display loading statistics
    stats = vector_store.get_stats()
    print(f"\n📊 Data Loading Summary:")
    print(f"   Total documents: {stats['total_documents']}")
    print(f"   Document types: {list(stats['document_types'].keys())}")


## Run Comprehensive Evaluation


In [ ]:
# Initialize the evaluator
print("🧪 Initializing RAGAS evaluator...")
evaluator = K8sRAGEvaluator(vector_store)

# Run comprehensive evaluation of all retrieval methods
print("🚀 Running comprehensive evaluation of all retrieval methods...")
print("This may take several minutes...")

comparison_df, summary = evaluator.run_comprehensive_evaluation()

if not comparison_df.empty:
    print("\n✅ Comprehensive evaluation completed!")
    print("\n📊 Retrieval Methods Comparison:")
    display(comparison_df)
else:
    print("❌ Comprehensive evaluation failed!")
